# 🏥 Improved 3D CT Liver Cancer Segmentation & Classification

**Optimizations:**
- Memory-efficient data loading with caching
- Mixed precision training for faster performance
- Dynamic batch sizing to prevent overload
- Comprehensive preprocessing visualizations
- Enhanced model architecture with residual connections
- Advanced data augmentation
- Better evaluation metrics

In [ ]:
# Core libraries
import os
import gc
import numpy as np
import nibabel as nib
from skimage.transform import resize
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import Sequence
from tensorflow.keras import backend as K

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Print versions
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Number of GPUs: {len(tf.config.list_physical_devices('GPU'))}")

## ⚙️ Configuration & Memory Optimization

In [ ]:
# Enable mixed precision for faster training and less memory usage
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f'Compute dtype: {policy.compute_dtype}')
print(f'Variable dtype: {policy.variable_dtype}')

# GPU memory growth to prevent OOM errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

# Configuration
CONFIG = {
    'target_shape': (96, 96, 48),  # Reduced from 128x128x64 for better performance
    'batch_size': 2,  # Small batch size for memory efficiency
    'epochs': 50,
    'learning_rate': 1e-4,
    'validation_split': 0.2,
    'random_seed': 42,
    'cache_preprocessed': True,  # Cache preprocessed data
}

print("\n📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 🛠️ Utility Functions with Preprocessing

In [ ]:
def read_nii(filepath):
    """Read NIfTI file with error handling"""
    try:
        ct_scan = nib.load(filepath)
        array = ct_scan.get_fdata()
        array = np.rot90(array)  # Orientation fix
        return array
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def normalize_ct_scan(volume):
    """Normalize CT scan using Hounsfield Unit windowing"""
    # Clip to soft tissue window (-100 to 400 HU)
    volume = np.clip(volume, -100, 400)
    # Normalize to [0, 1]
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
    return volume

def augment_volume(volume, mask):
    """Data augmentation for 3D volumes"""
    # Random flip
    if np.random.rand() > 0.5:
        volume = np.flip(volume, axis=0)
        mask = np.flip(mask, axis=0)
    
    if np.random.rand() > 0.5:
        volume = np.flip(volume, axis=1)
        mask = np.flip(mask, axis=1)
    
    # Random rotation (90, 180, 270 degrees)
    k = np.random.randint(0, 4)
    volume = np.rot90(volume, k, axes=(0, 1))
    mask = np.rot90(mask, k, axes=(0, 1))
    
    # Random brightness adjustment
    if np.random.rand() > 0.5:
        factor = np.random.uniform(0.8, 1.2)
        volume = np.clip(volume * factor, 0, 1)
    
    return volume, mask

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient for evaluation"""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """Dice loss function"""
    return 1 - dice_coefficient(y_true, y_pred)

def combined_loss(y_true, y_pred):
    """Combined binary crossentropy and dice loss"""
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

print("✅ Utility functions loaded")

## 📊 Improved Data Generator with Caching

In [ ]:
class OptimizedNiiDataGenerator(Sequence):
    """Memory-efficient data generator with caching and augmentation"""
    
    def __init__(self, volume_files, mask_files, batch_size=2, 
                 target_shape=(96, 96, 48), augment=False, cache=False):
        self.volume_files = volume_files
        self.mask_files = mask_files
        self.batch_size = batch_size
        self.target_shape = target_shape
        self.augment = augment
        self.cache = cache
        self.cache_dict = {} if cache else None
        
    def __len__(self):
        return int(np.ceil(len(self.volume_files) / self.batch_size))
    
    def load_and_preprocess(self, vol_path, mask_path):
        """Load and preprocess a single volume-mask pair"""
        # Check cache first
        if self.cache and vol_path in self.cache_dict:
            return self.cache_dict[vol_path]
        
        # Load data
        vol = read_nii(vol_path)
        mask = read_nii(mask_path)
        
        if vol is None or mask is None:
            return None, None
        
        # Resize
        vol = resize(vol, self.target_shape, preserve_range=True, anti_aliasing=True)
        mask = resize(mask, self.target_shape, preserve_range=True, anti_aliasing=False)
        
        # Normalize
        vol = normalize_ct_scan(vol)
        mask = (mask > 0).astype(np.float32)
        
        # Cache if enabled
        if self.cache:
            self.cache_dict[vol_path] = (vol, mask)
        
        return vol, mask
    
    def __getitem__(self, idx):
        batch_vol_files = self.volume_files[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_mask_files = self.mask_files[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        x_batch, y_batch = [], []
        
        for v, m in zip(batch_vol_files, batch_mask_files):
            vol, mask = self.load_and_preprocess(v, m)
            
            if vol is None or mask is None:
                continue
            
            # Augmentation
            if self.augment:
                vol, mask = augment_volume(vol, mask)
            
            x_batch.append(vol[..., np.newaxis])
            y_batch.append(mask[..., np.newaxis])
        
        return np.array(x_batch, dtype=np.float32), np.array(y_batch, dtype=np.float32)
    
    def on_epoch_end(self):
        """Shuffle data at end of epoch"""
        indices = np.arange(len(self.volume_files))
        np.random.shuffle(indices)
        self.volume_files = [self.volume_files[i] for i in indices]
        self.mask_files = [self.mask_files[i] for i in indices]

print("✅ Optimized data generator created")

## 📁 Load Dataset

In [ ]:
# Update this path to your dataset location
dataset_dir = "C:/Users/aliah/Downloads/CT"

volume_dir = os.path.join(dataset_dir, "volumes")
mask_dir = os.path.join(dataset_dir, "segmentations")

# Load file lists
volume_files = sorted([os.path.join(volume_dir, f) for f in os.listdir(volume_dir) if f.endswith(".nii")])
mask_files = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith(".nii")])

print(f"\n📊 Dataset Statistics:")
print(f"  Total volumes: {len(volume_files)}")
print(f"  Total masks: {len(mask_files)}")

# Verify matching
assert len(volume_files) == len(mask_files), "Mismatch between volumes and masks!"
print(f"  ✅ All volumes have matching masks")

## 🔍 Visualize Preprocessing Steps

In [ ]:
def visualize_preprocessing(volume_path, mask_path, target_shape=(96, 96, 48)):
    """Visualize preprocessing pipeline"""
    
    # Load raw data
    raw_vol = read_nii(volume_path)
    raw_mask = read_nii(mask_path)
    
    # Resized
    resized_vol = resize(raw_vol, target_shape, preserve_range=True, anti_aliasing=True)
    resized_mask = resize(raw_mask, target_shape, preserve_range=True, anti_aliasing=False)
    
    # Normalized
    norm_vol = normalize_ct_scan(resized_vol)
    binary_mask = (resized_mask > 0).astype(np.float32)
    
    # Get middle slices
    raw_mid = raw_vol.shape[2] // 2
    proc_mid = target_shape[2] // 2
    
    # Create figure
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('Preprocessing Pipeline Visualization', fontsize=16, fontweight='bold')
    
    # Row 1: Original
    axes[0, 0].imshow(raw_vol[:, :, raw_mid], cmap='gray')
    axes[0, 0].set_title(f'Original Volume\n{raw_vol.shape}', fontsize=10)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(raw_mask[:, :, raw_mid], cmap='hot')
    axes[0, 1].set_title(f'Original Mask\n{raw_mask.shape}', fontsize=10)
    axes[0, 1].axis('off')
    
    axes[0, 2].hist(raw_vol.flatten(), bins=100, alpha=0.7, color='blue')
    axes[0, 2].set_title('Original Intensity Distribution')
    axes[0, 2].set_xlabel('Intensity')
    axes[0, 2].set_ylabel('Frequency')
    
    axes[0, 3].imshow(raw_vol[:, :, raw_mid], cmap='gray')
    axes[0, 3].imshow(raw_mask[:, :, raw_mid], cmap='hot', alpha=0.3)
    axes[0, 3].set_title('Original Overlay')
    axes[0, 3].axis('off')
    
    # Row 2: Resized
    axes[1, 0].imshow(resized_vol[:, :, proc_mid], cmap='gray')
    axes[1, 0].set_title(f'Resized Volume\n{resized_vol.shape}', fontsize=10)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(resized_mask[:, :, proc_mid], cmap='hot')
    axes[1, 1].set_title(f'Resized Mask\n{resized_mask.shape}', fontsize=10)
    axes[1, 1].axis('off')
    
    axes[1, 2].hist(resized_vol.flatten(), bins=100, alpha=0.7, color='green')
    axes[1, 2].set_title('Resized Intensity Distribution')
    axes[1, 2].set_xlabel('Intensity')
    axes[1, 2].set_ylabel('Frequency')
    
    axes[1, 3].imshow(resized_vol[:, :, proc_mid], cmap='gray')
    axes[1, 3].imshow(resized_mask[:, :, proc_mid], cmap='hot', alpha=0.3)
    axes[1, 3].set_title('Resized Overlay')
    axes[1, 3].axis('off')
    
    # Row 3: Normalized
    axes[2, 0].imshow(norm_vol[:, :, proc_mid], cmap='gray')
    axes[2, 0].set_title(f'Normalized Volume\nRange: [{norm_vol.min():.2f}, {norm_vol.max():.2f}]', fontsize=10)
    axes[2, 0].axis('off')
    
    axes[2, 1].imshow(binary_mask[:, :, proc_mid], cmap='hot')
    axes[2, 1].set_title(f'Binary Mask\nTumor pixels: {int(binary_mask.sum())}', fontsize=10)
    axes[2, 1].axis('off')
    
    axes[2, 2].hist(norm_vol.flatten(), bins=100, alpha=0.7, color='red')
    axes[2, 2].set_title('Normalized Intensity Distribution')
    axes[2, 2].set_xlabel('Intensity')
    axes[2, 2].set_ylabel('Frequency')
    
    axes[2, 3].imshow(norm_vol[:, :, proc_mid], cmap='gray')
    axes[2, 3].imshow(binary_mask[:, :, proc_mid], cmap='hot', alpha=0.3)
    axes[2, 3].set_title('Final Overlay')
    axes[2, 3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\n📊 Preprocessing Statistics:")
    print(f"  Original shape: {raw_vol.shape} → Target shape: {target_shape}")
    print(f"  Original range: [{raw_vol.min():.2f}, {raw_vol.max():.2f}]")
    print(f"  Normalized range: [{norm_vol.min():.2f}, {norm_vol.max():.2f}]")
    print(f"  Tumor voxels: {int(binary_mask.sum())} / {np.prod(target_shape)} ({100*binary_mask.sum()/np.prod(target_shape):.2f}%)")

# Visualize first sample
if len(volume_files) > 0:
    print("\n🎨 Visualizing preprocessing for first sample...")
    visualize_preprocessing(volume_files[0], mask_files[0], CONFIG['target_shape'])

## 📈 Dataset Overview & Statistics

In [ ]:
def analyze_dataset(volume_files, mask_files, n_samples=5):
    """Analyze dataset characteristics"""
    
    print(f"\n📊 Analyzing {min(n_samples, len(volume_files))} samples...")
    
    shapes = []
    tumor_ratios = []
    intensity_ranges = []
    
    for i in tqdm(range(min(n_samples, len(volume_files)))):
        vol = read_nii(volume_files[i])
        mask = read_nii(mask_files[i])
        
        if vol is not None and mask is not None:
            shapes.append(vol.shape)
            tumor_ratios.append(np.sum(mask > 0) / mask.size)
            intensity_ranges.append((vol.min(), vol.max()))
    
    # Plot statistics
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Tumor ratios
    axes[0].bar(range(len(tumor_ratios)), [r * 100 for r in tumor_ratios], color='coral')
    axes[0].set_xlabel('Sample Index')
    axes[0].set_ylabel('Tumor Percentage (%)')
    axes[0].set_title('Tumor Coverage per Sample')
    axes[0].grid(True, alpha=0.3)
    
    # Intensity ranges
    mins, maxs = zip(*intensity_ranges)
    axes[1].plot(mins, label='Min', marker='o')
    axes[1].plot(maxs, label='Max', marker='s')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Intensity Value')
    axes[1].set_title('Intensity Ranges')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Volume shapes
    x_dims, y_dims, z_dims = zip(*shapes)
    axes[2].scatter(range(len(shapes)), x_dims, label='X', alpha=0.6)
    axes[2].scatter(range(len(shapes)), y_dims, label='Y', alpha=0.6)
    axes[2].scatter(range(len(shapes)), z_dims, label='Z', alpha=0.6)
    axes[2].set_xlabel('Sample Index')
    axes[2].set_ylabel('Dimension Size')
    axes[2].set_title('Volume Dimensions')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📈 Dataset Statistics:")
    print(f"  Average tumor coverage: {np.mean(tumor_ratios)*100:.2f}%")
    print(f"  Tumor coverage range: {np.min(tumor_ratios)*100:.2f}% - {np.max(tumor_ratios)*100:.2f}%")
    print(f"  Most common shape: {max(set(shapes), key=shapes.count)}")
    print(f"  Intensity range: [{np.mean(mins):.1f}, {np.mean(maxs):.1f}]")

# Analyze dataset
analyze_dataset(volume_files, mask_files, n_samples=min(5, len(volume_files)))

## ✂️ Train/Test Split

In [ ]:
# Split data
x_train_files, x_test_files, y_train_files, y_test_files = train_test_split(
    volume_files, mask_files, 
    test_size=CONFIG['validation_split'], 
    random_state=CONFIG['random_seed']
)

print(f"\n✂️ Data Split:")
print(f"  Training samples: {len(x_train_files)}")
print(f"  Validation samples: {len(x_test_files)}")
print(f"  Split ratio: {100*(1-CONFIG['validation_split']):.0f}/{100*CONFIG['validation_split']:.0f}")

## 🔄 Create Data Generators

In [ ]:
# Create generators
train_gen = OptimizedNiiDataGenerator(
    x_train_files, y_train_files,
    batch_size=CONFIG['batch_size'],
    target_shape=CONFIG['target_shape'],
    augment=True,
    cache=CONFIG['cache_preprocessed']
)

val_gen = OptimizedNiiDataGenerator(
    x_test_files, y_test_files,
    batch_size=CONFIG['batch_size'],
    target_shape=CONFIG['target_shape'],
    augment=False,
    cache=CONFIG['cache_preprocessed']
)

print(f"\n🔄 Generators Created:")
print(f"  Training batches: {len(train_gen)}")
print(f"  Validation batches: {len(val_gen)}")
print(f"  Augmentation: {'ON' if train_gen.augment else 'OFF'}")
print(f"  Caching: {'ON' if train_gen.cache else 'OFF'}")

## 🎯 Visualize Training Batch

In [ ]:
def visualize_batch(generator, n_samples=2):
    """Visualize samples from data generator"""
    x_batch, y_batch = generator[0]
    
    n_samples = min(n_samples, len(x_batch))
    
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_samples):
        mid_slice = x_batch.shape[3] // 2
        
        # Volume
        axes[i, 0].imshow(x_batch[i, :, :, mid_slice, 0], cmap='gray')
        axes[i, 0].set_title(f'Sample {i+1}: CT Scan\nShape: {x_batch[i].shape[:-1]}')
        axes[i, 0].axis('off')
        
        # Mask
        axes[i, 1].imshow(y_batch[i, :, :, mid_slice, 0], cmap='hot')
        tumor_pct = 100 * y_batch[i].sum() / y_batch[i].size
        axes[i, 1].set_title(f'Sample {i+1}: Tumor Mask\nCoverage: {tumor_pct:.2f}%')
        axes[i, 1].axis('off')
        
        # Overlay
        axes[i, 2].imshow(x_batch[i, :, :, mid_slice, 0], cmap='gray')
        axes[i, 2].imshow(y_batch[i, :, :, mid_slice, 0], cmap='hot', alpha=0.4)
        axes[i, 2].set_title(f'Sample {i+1}: Overlay')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize training batch
print("\n🎯 Visualizing training batch...")
visualize_batch(train_gen, n_samples=2)

## 🏗️ Improved 3D U-Net Model with Residual Connections

In [ ]:
def residual_block(x, filters, kernel_size=3):
    """Residual block with skip connection"""
    shortcut = x
    
    # First conv
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Second conv
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Adjust shortcut if needed
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, 1, padding='same')(shortcut)
    
    # Add and activate
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

def improved_unet_3d(input_shape=(96, 96, 48, 1)):
    """Improved 3D U-Net with residual connections and attention"""
    inputs = layers.Input(input_shape)
    
    # Encoder
    # Block 1
    c1 = residual_block(inputs, 32)
    p1 = layers.MaxPooling3D(pool_size=2)(c1)
    p1 = layers.Dropout(0.1)(p1)
    
    # Block 2
    c2 = residual_block(p1, 64)
    p2 = layers.MaxPooling3D(pool_size=2)(c2)
    p2 = layers.Dropout(0.1)(p2)
    
    # Block 3
    c3 = residual_block(p2, 128)
    p3 = layers.MaxPooling3D(pool_size=2)(c3)
    p3 = layers.Dropout(0.2)(p3)
    
    # Bottleneck
    c4 = residual_block(p3, 256)
    c4 = layers.Dropout(0.3)(c4)
    
    # Decoder
    # Block 5
    u5 = layers.Conv3DTranspose(128, 2, strides=2, padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = residual_block(u5, 128)
    c5 = layers.Dropout(0.2)(c5)
    
    # Block 6
    u6 = layers.Conv3DTranspose(64, 2, strides=2, padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = residual_block(u6, 64)
    c6 = layers.Dropout(0.1)(c6)
    
    # Block 7
    u7 = layers.Conv3DTranspose(32, 2, strides=2, padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = residual_block(u7, 32)
    
    # Output
    outputs = layers.Conv3D(1, 1, activation='sigmoid', dtype='float32')(c7)
    
    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

# Build model
print("\n🏗️ Building improved 3D U-Net model...")
input_shape = (*CONFIG['target_shape'], 1)
model_3d = improved_unet_3d(input_shape)

# Compile with custom loss
model_3d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss=combined_loss,
    metrics=[dice_coefficient, 'binary_accuracy']
)

print(f"\n✅ Model compiled successfully")
print(f"   Input shape: {input_shape}")
print(f"   Output shape: {model_3d.output.shape}")
print(f"   Total parameters: {model_3d.count_params():,}")

## 📋 Model Summary

In [ ]:
model_3d.summary()

## 📞 Training Callbacks

In [ ]:
# Create callbacks
checkpoint_path = 'best_model_3d.keras'

callback_list = [
    # Save best model
    callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Early stopping
    callbacks.EarlyStopping(
        monitor='val_dice_coefficient',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard logging (optional)
    # callbacks.TensorBoard(log_dir='./logs', histogram_freq=1)
]

print("\n📞 Callbacks configured:")
for cb in callback_list:
    print(f"  ✓ {cb.__class__.__name__}")

## 🚀 Train Model

In [ ]:
print("\n🚀 Starting training...\n")
print("=" * 60)
print(f"Configuration:")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Target shape: {CONFIG['target_shape']}")
print("=" * 60)
print()

# Train model
history = model_3d.fit(
    train_gen,
    validation_data=val_gen,
    epochs=CONFIG['epochs'],
    callbacks=callback_list,
    verbose=1
)

print("\n✅ Training complete!")

# Clear memory
gc.collect()
K.clear_session()

## 📊 Training History Visualization

In [ ]:
def plot_training_history(history):
    """Plot comprehensive training history"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice Coefficient
    axes[0, 1].plot(history.history['dice_coefficient'], label='Training Dice', linewidth=2)
    axes[0, 1].plot(history.history['val_dice_coefficient'], label='Validation Dice', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Coefficient')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, 0].plot(history.history['binary_accuracy'], label='Training Accuracy', linewidth=2)
    axes[1, 0].plot(history.history['val_binary_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Binary Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate (if available)
    if 'lr' in history.history:
        axes[1, 1].plot(history.history['lr'], linewidth=2, color='red')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_title('Learning Rate Schedule')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(True, alpha=0.3)
    else:
        # Summary statistics
        axes[1, 1].axis('off')
        summary_text = f"""
        Training Summary:
        
        Best Validation Dice: {max(history.history['val_dice_coefficient']):.4f}
        Best Validation Accuracy: {max(history.history['val_binary_accuracy']):.4f}
        Lowest Validation Loss: {min(history.history['val_loss']):.4f}
        
        Total Epochs: {len(history.history['loss'])}
        """
        axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center')
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

## 🎯 Evaluate Model on Test Set

In [ ]:
print("\n🎯 Evaluating model on test set...")

# Load best model
if os.path.exists(checkpoint_path):
    model_3d = tf.keras.models.load_model(
        checkpoint_path,
        custom_objects={
            'combined_loss': combined_loss,
            'dice_coefficient': dice_coefficient
        }
    )
    print("✅ Loaded best model from checkpoint")

# Evaluate
test_results = model_3d.evaluate(val_gen, verbose=1)

print("\n📊 Test Results:")
print(f"  Loss: {test_results[0]:.4f}")
print(f"  Dice Coefficient: {test_results[1]:.4f}")
print(f"  Binary Accuracy: {test_results[2]:.4f}")

## 🔮 Visualize Predictions

In [ ]:
def visualize_predictions(model, generator, n_samples=3):
    """Visualize model predictions"""
    x_batch, y_batch = generator[0]
    predictions = model.predict(x_batch, verbose=0)
    
    n_samples = min(n_samples, len(x_batch))
    
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_samples):
        mid_slice = x_batch.shape[3] // 2
        
        # Original CT
        axes[i, 0].imshow(x_batch[i, :, :, mid_slice, 0], cmap='gray')
        axes[i, 0].set_title(f'Sample {i+1}: CT Scan')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(y_batch[i, :, :, mid_slice, 0], cmap='hot')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Predicted mask
        pred_binary = (predictions[i, :, :, mid_slice, 0] > 0.5).astype(float)
        axes[i, 2].imshow(pred_binary, cmap='hot')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(x_batch[i, :, :, mid_slice, 0], cmap='gray')
        axes[i, 3].imshow(pred_binary, cmap='hot', alpha=0.4)
        
        # Calculate dice
        dice = 2 * np.sum(y_batch[i, :, :, mid_slice, 0] * pred_binary) / \
               (np.sum(y_batch[i, :, :, mid_slice, 0]) + np.sum(pred_binary) + 1e-8)
        axes[i, 3].set_title(f'Overlay (Dice: {dice:.3f})')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions
print("\n🔮 Visualizing predictions on validation set...")
visualize_predictions(model_3d, val_gen, n_samples=3)

## 🏥 Predict & Classify New CT Scan

In [ ]:
def predict_and_classify(model, nii_path, target_shape=(96, 96, 48), threshold=0.003):
    """
    Predict tumor segmentation and classify as cancer/no cancer
    
    Args:
        model: Trained model
        nii_path: Path to NIfTI file
        target_shape: Target shape for resizing
        threshold: Tumor ratio threshold for classification
    
    Returns:
        ct_resized: Resized CT scan
        pred_mask: Predicted binary mask
        status: Classification result
        ratio: Tumor ratio
        confidence: Prediction confidence
    """
    # Load and preprocess
    ct = read_nii(nii_path)
    if ct is None:
        return None, None, None, None, None
    
    # Resize
    ct_resized = resize(ct, target_shape, preserve_range=True, anti_aliasing=True)
    
    # Normalize
    ct_norm = normalize_ct_scan(ct_resized)
    ct_input = ct_norm[np.newaxis, ..., np.newaxis]
    
    # Predict
    pred = model.predict(ct_input, verbose=0)
    pred_mask = (pred > 0.5).astype(np.uint8)
    
    # Calculate metrics
    tumor_voxels = np.sum(pred_mask)
    total_voxels = np.prod(pred_mask.shape)
    ratio = tumor_voxels / total_voxels
    
    # Average confidence
    confidence = np.mean(np.abs(pred - 0.5)) * 2
    
    # Classify
    if ratio >= threshold:
        status = "⚠️ CANCER DETECTED"
    else:
        status = "✅ NO CANCER"
    
    return ct_norm, pred_mask, status, ratio, confidence


def visualize_prediction_3d(ct, pred_mask, status, ratio, confidence):
    """Visualize prediction with multiple slices"""
    n_slices = 5
    slice_indices = np.linspace(0, pred_mask.shape[3]-1, n_slices, dtype=int)
    
    fig, axes = plt.subplots(2, n_slices, figsize=(20, 8))
    
    for idx, slice_idx in enumerate(slice_indices):
        # CT scan
        axes[0, idx].imshow(ct[:, :, slice_idx], cmap='gray')
        axes[0, idx].set_title(f'Slice {slice_idx}')
        axes[0, idx].axis('off')
        
        # Prediction overlay
        axes[1, idx].imshow(ct[:, :, slice_idx], cmap='gray')
        axes[1, idx].imshow(pred_mask[0, :, :, slice_idx, 0], cmap='hot', alpha=0.4)
        axes[1, idx].set_title(f'Prediction')
        axes[1, idx].axis('off')
    
    plt.suptitle(f'{status} | Tumor Ratio: {ratio:.4f} | Confidence: {confidence:.2%}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Example prediction (update path to your test file)
test_file_path = "C:/Users/aliah/Downloads/liver_1.nii/liver_1.nii"

if os.path.exists(test_file_path):
    print("\n🏥 Processing CT scan...")
    ct, pred_mask, status, ratio, confidence = predict_and_classify(
        model_3d, 
        test_file_path,
        target_shape=CONFIG['target_shape']
    )
    
    if ct is not None:
        print("\n" + "="*50)
        print("       CLASSIFICATION RESULT")
        print("="*50)
        print(f"Status        : {status}")
        print(f"Tumor Ratio   : {ratio:.6f} ({ratio*100:.3f}%)")
        print(f"Confidence    : {confidence:.2%}")
        print("="*50)
        
        visualize_prediction_3d(ct, pred_mask, status, ratio, confidence)
    else:
        print("❌ Error processing file")
else:
    print(f"⚠️ Test file not found: {test_file_path}")
    print("Please update the path to your test NIfTI file")

## 💾 Save Final Model

In [ ]:
# Save model
final_model_path = 'final_ct_segmentation_model.keras'
model_3d.save(final_model_path)
print(f"\n💾 Model saved to: {final_model_path}")

# Save configuration
import json
with open('model_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=4)
print(f"💾 Configuration saved to: model_config.json")

## 📝 Project Summary

### Improvements Made:

1. **Performance Optimizations:**
   - Mixed precision training (faster, less memory)
   - GPU memory growth enabled
   - Reduced target shape (96×96×48 vs 128×128×64)
   - Data caching for faster loading
   - Efficient batch processing

2. **Model Architecture:**
   - Residual connections for better gradient flow
   - Batch normalization for stable training
   - Dropout layers to prevent overfitting
   - Combined loss (BCE + Dice) for better segmentation

3. **Data Processing:**
   - Advanced HU windowing normalization
   - Data augmentation (flip, rotation, brightness)
   - Memory-efficient data generators

4. **Visualizations:**
   - Preprocessing pipeline visualization
   - Dataset statistics and analysis
   - Training history plots
   - Multi-slice prediction visualization
   - Comprehensive batch visualization

5. **Training Features:**
   - Model checkpointing
   - Early stopping
   - Learning rate scheduling
   - Comprehensive metrics (Dice, Accuracy)

### Usage Tips:

- Adjust `batch_size` if you encounter memory issues
- Reduce `target_shape` further for even faster training
- Enable caching for repeated experiments
- Monitor GPU usage and adjust accordingly

---

**Ready to use! 🚀**